In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from PIL import Image
import glob
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Define paths and classes
dataset_path = 'C:/Users/murug/.cache/kagglehub/datasets/feyzazkefe/trashnet/versions/1/dataset-resized'  # Update with your dataset path after unzipping
classes = ['plastic', 'metal', 'glass', 'cardboard', 'paper']
target_size = (224, 224)  # Resize dimensions
batch_size = 32
num_classes = len(classes)
epochs = 5  # Adjust as needed

In [ ]:
# 1. Data Preparation
def load_dataset():
    print("Loading Dataset...")
    image_paths = []
    labels = []
    
    # Collect image paths and labels
    for cls in classes:
        paths = glob.glob(os.path.join(dataset_path, cls, '*.jpg'))  # Adjust extension if needed
        image_paths.extend(paths)
        labels.extend([cls] * len(paths))
    
    # Check for corrupted images
    cleaned_paths = []
    cleaned_labels = []
    for img_path, label in zip(image_paths, labels):
        try:
            img = Image.open(img_path)
            img.verify()
            cleaned_paths.append(img_path)
            cleaned_labels.append(label)
        except:
            print(f'Removing corrupted image: {img_path}')
    
    print(f'Total images after cleaning: {len(cleaned_paths)}')
    
    # Split dataset (70% train, 15% val, 15% test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        cleaned_paths, cleaned_labels, test_size=0.3, stratify=cleaned_labels, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )
    
    print(f'Training samples: {len(X_train)}')
    print(f'Validation samples: {len(X_val)}')
    print(f'Test samples: {len(X_test)}')
    
    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
# 2. Data Cleaning & Preprocessing
def handle_class_imbalance(X_train, y_train):
    print("Checking for class imbalance...")
    class_counts = Counter(y_train)
    print("Class distribution before balancing:", class_counts)
    
    # Convert labels to numerical indices
    label_to_index = {cls: idx for idx, cls in enumerate(classes)}
    y_train_numeric = np.array([label_to_index[label] for label in y_train])
    
    # Reshape for RandomOverSampler
    X_train_array = np.array(X_train).reshape(-1, 1)
    
    # Oversample minority classes
    ros = RandomOverSampler(random_state=42)
    X_resampled, y_resampled = ros.fit_resample(X_train_array, y_train_numeric)
    
    # Convert back to original format
    X_train_balanced = X_resampled.flatten().tolist()
    y_train_balanced = [classes[idx] for idx in y_resampled]
    
    print("Class distribution after balancing:", Counter(y_train_balanced))
    return X_train_balanced, y_train_balanced

def create_data_generator(X, y, data_type='train'):
    if data_type == 'train':
        datagen = ImageDataGenerator(
            rescale=1./255,  # Normalize pixel values
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            horizontal_flip=True,
            zoom_range=0.2,
            fill_mode='nearest'
        )
    else:
        datagen = ImageDataGenerator(rescale=1./255)  # Only normalize for val/test
    
    # Load images in batches, ensuring 3 channels (RGB)
    def load_image(img_path):
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)  # Force RGB loading
        if img is None:
            raise ValueError(f"Failed to load image: {img_path}")
        img = cv2.resize(img, target_size)  # Resize to 224x224
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Ensure RGB format
        if img.shape[-1] != 3:  # Additional check for channel count
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)  # Convert grayscale to RGB if needed
        if img.shape != (224, 224, 3):  # Verify final shape
            raise ValueError(f"Image {img_path} has incorrect shape: {img.shape}")
        return img
    
    # Create generator
    images = []
    for path in X:
        img = load_image(path)
        images.append(img)
    images = np.array(images)
    labels = np.array([classes.index(label) for label in y])
    
    # Verify image shapes
    for i, img in enumerate(images):
        if img.shape != (224, 224, 3):
            raise ValueError(f"Image at index {i} has incorrect shape: {img.shape}")
    
    generator = datagen.flow(
        images, labels,
        batch_size=batch_size,
        shuffle=(data_type == 'train')
    )
    return generator, images, labels

In [ ]:
# 3. Exploratory Data Analysis (EDA)
def perform_eda(X_train, y_train):
    print("Performing EDA...")
    
    # Visualize number of images per class
    class_counts = Counter(y_train)
    plt.figure(figsize=(8, 5))
    plt.bar(class_counts.keys(), class_counts.values(), color='skyblue')
    plt.title('Number of Images per Class')
    plt.xlabel('Class')
    plt.ylabel('Number of Images')
    plt.xticks(rotation=45)
    plt.show()
    
    # Show example images from each category
    plt.figure(figsize=(15, 3))
    for i, cls in enumerate(classes):
        cls_paths = [path for path, label in zip(X_train, y_train) if label == cls]
        if cls_paths:
            img_path = cls_paths[0]
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(1, len(classes), i+1)
            plt.imshow(img)
            plt.title(cls)
            plt.axis('off')
    plt.show()
    
    # Analyze pixel intensity/color distribution
    plt.figure(figsize=(12, 4))
    for i, channel in enumerate(['Red', 'Green', 'Blue']):
        pixel_values = []
        for path in X_train[:100]:  # Sample 100 images for efficiency
            img = cv2.imread(path)
            img = cv2.resize(img, target_size)
            pixel_values.append(img[:, :, i].ravel())  # Get pixel values for channel
        pixel_values = np.concatenate(pixel_values)
        
        plt.subplot(1, 3, i+1)
        plt.hist(pixel_values, bins=50, color=channel.lower(), alpha=0.7)
        plt.title(f'{channel} Channel Distribution')
        plt.xlabel('Pixel Intensity')
        plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()


In [ ]:
# 4. Model Development
def build_transfer_learning_model(model_name):
    print(f"Building {model_name} Model...")
    
    # Load base model
    if model_name == 'ResNet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    elif model_name == 'MobileNetV2':
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    elif model_name == 'EfficientNetB0':
        try:
            base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        except Exception as e:
            print(f"Error loading EfficientNetB0 weights: {e}")
            print("Falling back to initializing EfficientNetB0 without pre-trained weights.")
            base_model = EfficientNetB0(weights=None, include_top=False, input_shape=(224, 224, 3))
    else:
        raise ValueError("Unsupported model name")
    
    # Freeze base layers
    for layer in base_model.layers:
        layer.trainable = False
    
    # Build model
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    
    # Compile model
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    model.summary()
    return model

In [ ]:
# 5. Model Evaluation
def evaluate_model(model, test_generator, X_test, y_test, model_name):
    print(f"Evaluating {model_name}...")
    
    # Predict on test set
    test_images, test_labels = test_generator[1], test_generator[2]
    y_pred = model.predict(test_images)
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(test_labels, y_pred_classes)
    precision = precision_score(test_labels, y_pred_classes, average='weighted')
    recall = recall_score(test_labels, y_pred_classes, average='weighted')
    f1 = f1_score(test_labels, y_pred_classes, average='weighted')
    
    print(f"{model_name} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(test_labels, y_pred_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    
    # Visualize misclassifications (up to 5 examples)
    misclassified_idx = np.where(y_pred_classes != test_labels)[0]
    if len(misclassified_idx) > 0:
        print(f"Misclassified Examples for {model_name}:")
        plt.figure(figsize=(15, 3))
        for i, idx in enumerate(misclassified_idx[:5]):  # Show up to 5
            img = test_images[idx]
            true_label = classes[test_labels[idx]]
            pred_label = classes[y_pred_classes[idx]]
            plt.subplot(1, min(5, len(misclassified_idx)), i+1)
            plt.imshow(img.astype(np.uint8))
            plt.title(f'True: {true_label}\nPred: {pred_label}')
            plt.axis('off')
        plt.show()
    
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

In [ ]:
# 6. Best Model Selection
def select_best_model(models_metrics):
    print("Selecting Best Model...")
    best_model = max(models_metrics, key=lambda x: x['metrics']['f1'])
    print(f"Best Model: {best_model['name']}")
    print(f"F1-Score: {best_model['metrics']['f1']:.4f}")
    print(f"Accuracy: {best_model['metrics']['accuracy']:.4f}")
    print(f"Precision: {best_model['metrics']['precision']:.4f}")
    print(f"Recall: {best_model['metrics']['recall']:.4f}")
    return best_model

In [ ]:
def main():
    # Step 1: Load dataset
    X_train, y_train, X_val, y_val, X_test, y_test = load_dataset()
    
    # Step 2: Perform EDA
    perform_eda(X_train, y_train)
    
    # Step 3: Handle class imbalance
    X_train_balanced, y_train_balanced = handle_class_imbalance(X_train, y_train)
    
    # Step 4: Create data generators
    train_generator, _, _ = create_data_generator(X_train_balanced, y_train_balanced, 'train')
    val_generator, _, _ = create_data_generator(X_val, y_val, 'val')
    test_generator, test_images, test_labels = create_data_generator(X_test, y_test, 'test')
    
    print("Data preparation and preprocessing completed!")
    print(f"Training generator: {len(X_train_balanced)} samples")
    print(f"Validation generator: {len(X_val)} samples")
    print(f"Test generator: {len(X_test)} samples")
    
    # Step 5: Train and evaluate models
    model_names = ['ResNet50', 'MobileNetV2', 'EfficientNetB0']
    models_metrics = []
    
    for model_name in model_names:
        # Build and train model
        model = build_transfer_learning_model(model_name)
        history = model.fit(
            train_generator,
            steps_per_epoch=len(X_train_balanced) // batch_size,
            validation_data=val_generator,
            validation_steps=len(X_val) // batch_size,
            epochs=epochs,
            verbose=1
        )
        
        # Evaluate model
        metrics = evaluate_model(model, (None, test_images, test_labels), X_test, y_test, model_name)
        models_metrics.append({'name': model_name, 'model': model, 'metrics': metrics})
        
        # Save model
        model.save(f'trashnet_{model_name.lower()}.h5')
        
        # Plot training history
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(history.history['accuracy'], label='Train Accuracy')
        plt.plot(history.history['val_accuracy'], label='Val Accuracy')
        plt.title(f'{model_name} Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        plt.plot(history.history['loss'], label='Train Loss')
        plt.plot(history.history['val_loss'], label='Val Loss')
        plt.title(f'{model_name} Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.show()

In [ ]:
# Run main to get all necessary variables
best_model, train_generator, val_generator, test_generator = main()

# Save data paths
np.save('train_paths.npy', X_train_balanced)
np.save('train_labels.npy', y_train_balanced)
np.save('val_paths.npy', X_val)
np.save('val_labels.npy', y_val)
np.save('test_paths.npy', X_test)
np.save('test_labels.npy', y_test)

In [ ]:
if __name__ == "__main__":
    best_model, train_generator, val_generator, test_generator = main()
    
    # Verify batch
    X_batch, y_batch = next(train_generator)
    print(f"Batch shape: {X_batch.shape}, Labels shape: {y_batch.shape}")